# Activity 14: Sentiment Analysis on Real Reviews

Amazon Digital Video Games reviews, evaluated against star ratings.

## Part 1: Load and Explore the Data

### Step 2: Load the data

In [1]:
import pandas as pd

DATA_PATH = "data/amazon_reviews_us_Digital_Video_Games_v1_00.tsv.zip"

df = pd.read_csv(DATA_PATH, sep="\t")
print("Default settings:", df.shape)

Default settings: (144724, 15)


**How many rows and columns did pandas load?**

pandas loaded **144,724 rows and 15 columns**.

### Step 3: Load it again, correctly

In [2]:
df = pd.read_csv(DATA_PATH, sep="\t", quoting=3)  # 3 = treat quotes as plain text
print("quoting=3:", df.shape)

quoting=3: (145431, 15)


**How many rows does each version load? Did pandas warn you about the difference?**

**What does that tell you about checking your data?**

Default settings loaded **144,724 rows**; with `quoting=3` it loaded **145,431 rows**, so the default load silently dropped **707 reviews**. No, pandas gave no warning at all. It tells me to always check the shape of my data against what I expect (and compare loading options) before analyzing, because a load can "succeed" while quietly losing or merging rows.

### Step 4: Read some actual reviews

In [3]:
peek = df["review_body"].sample(5, random_state=2134)
for text in peek:
    print(text, "\n")

The &#34;AI&#34; is wrong by cheating, many bugs and control system cheating is not reliable<br />the PC version is penalized 

Is fast secure and very easy you only payed and in less than a minute you can use the code thanks 

This is exactly what it says it is, and I was able to email it as a gift!  Perfect.  Thanks! 

Since there is no way to return downloaded games or even remark on my problem, my only other options is via a review.<br /><br />I have a steam account and am logged in.  I tried installing the game, and it gave me a generic &#34;Trouble Communicating With Gateway&#34; message.  Googling the message returns one web page in German.  Retried three times with the same result.<br /><br />So...  Looks like I'm screwed. 

I bought this crapware online but was not able to install it on my tablet nor my android phone. The amazon android market app \\"seems\\" to crash when installing free or promotional apps. Amazon customer service is non existent. They dont care once they ha

**What do you notice that looks weird in these reviews?**

**Where do you think it came from?**

The reviews contain HTML junk: literal `<br />` tags in the middle of the text and `&#34;` in place of quotation marks ("The &#34;AI&#34; is wrong..."), plus doubled-up backslash-quotes (`\\"seems\\"`). It came from the website: reviews were typed into a web form and stored as HTML (line breaks became `<br />`, quotes became HTML entities, and quotes got escaped) when the dataset was exported.

### Step 5: Star ratings

In [4]:
df["star_rating"].value_counts(normalize=True).sort_index()

star_rating
1    0.171697
2    0.053283
3    0.079962
4    0.140314
5    0.554744
Name: proportion, dtype: float64

**How are the ratings distributed? Why might that matter when you measure a model's accuracy?**

The ratings are heavily skewed toward 5 stars: 1 star = 17.2%, 2 = 5.3%, 3 = 8.0%, 4 = 14.0%, 5 = 55.5%. This matters because a model that just predicted "5 stars" for everything would be right about 55% of the time, so overall accuracy would make a lazy model look decent. It also means there are few 2- and 3-star examples to judge the model on.

### Step 6: Products

In [5]:
df["product_title"].value_counts().head(10)

product_title
Playstation Network Card                                          13642
Xbox Live Subscription                                             7307
Playstation Plus Subscription                                      4712
Xbox Live Gift Card                                                3438
SimCity - Limited Edition                                          3421
Xbox 360 Live Points Card                                          3036
Battlefield 4                                                      1453
Final Fantasy XIV: A Realm Reborn                                  1273
Battlefield 3                                                      1160
Command and Conquer The Ultimate Collection [Online Game Code]      875
Name: count, dtype: int64

**What do you notice about the most-reviewed “games”?**

Most of the top "games" aren't games: they are gift cards, subscriptions, and points cards (Playstation Network Card with 13,642 reviews, Xbox Live Subscription, Playstation Plus Subscription, Xbox Live Gift Card, Xbox 360 Live Points Card). Their reviews are about getting the code quickly rather than about gameplay, and they make up a big share of the dataset.

### Step 7: Clean the text

In [6]:
print("Reviews with <br />:", df["review_body"].str.contains("<br />").sum())
df = df.dropna(subset=["review_body"])
df["review_body"] = df["review_body"].str.replace("<br />", " ")
print("After cleaning:", df["review_body"].str.contains("<br />").sum(), "\n")

for text in df.loc[peek.index, "review_body"]:
    print(text, "\n")

Reviews with <br />: 25699
After cleaning: 0 

The &#34;AI&#34; is wrong by cheating, many bugs and control system cheating is not reliable the PC version is penalized 

Is fast secure and very easy you only payed and in less than a minute you can use the code thanks 

This is exactly what it says it is, and I was able to email it as a gift!  Perfect.  Thanks! 

Since there is no way to return downloaded games or even remark on my problem, my only other options is via a review.  I have a steam account and am logged in.  I tried installing the game, and it gave me a generic &#34;Trouble Communicating With Gateway&#34; message.  Googling the message returns one web page in German.  Retried three times with the same result.  So...  Looks like I'm screwed. 

I bought this crapware online but was not able to install it on my tablet nor my android phone. The amazon android market app \\"seems\\" to crash when installing free or promotional apps. Amazon customer service is non existent. They 

**How many reviews had line breaks before cleaning, and after?**

**What still looks weird in the five reviews? Do you think it would change whether a model reads a review as positive or negative?**

**25,699** reviews had `<br />` before cleaning and **0** after (this also dropped the 3 empty reviews). Still weird: the `&#34;` HTML entities remain in place of quotation marks, the escaped `\\"` quotes are still there, and there are doubled spaces where the `<br />` tags were. I don't think they would change whether a model reads a review as positive or negative, since the sentiment words ("crapware", "Perfect", "screwed") are untouched and the leftover symbols are just noise, though they could slightly confuse the tokenizer.

### Step 8: Take a sample

In [7]:
sample = df.groupby("star_rating").sample(n=100, random_state=42)
complaints = df[df["star_rating"] == 1].sample(n=20, random_state=42)
print(len(sample), len(complaints))

500 20


**Why take the same number of each star rating instead of 500 random reviews? (Look at your answer in step 5.)**

Because ratings are so imbalanced (55% are 5 stars, only 5% are 2 stars), 500 random reviews would be mostly 5-star with very few 2- and 3-star examples, so I couldn't measure how the model does on those ratings, and the result would be dominated by the 5-star class. 100 of each star rating gives every rating equal weight, so accuracy is not inflated by the most common rating.

## Part 2: Run a Sentiment Model

### Step 9: Try one review

In [8]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)
print(sample["star_rating"].iloc[0], sample["review_body"].iloc[0][:200])
classifier(sample["review_body"].iloc[0])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

1 Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get 


[{'label': 'NEGATIVE', 'score': 0.9998071789741516}]

**What are the keys in the result, and what does the score mean?**

**Does the model agree with the customer's star rating?**

The result is a list with one dictionary with two keys: `label` (`NEGATIVE` or `POSITIVE`) and `score`. The score is the model's confidence in that label, from 0 to 1 (here 0.9998, so it is nearly certain the review is negative). Yes: the customer gave 1 star and the model said NEGATIVE, so they agree.

### Step 10: Classify all 500 reviews in one call

In [9]:
results = classifier(sample["review_body"].tolist(), truncation=True)
sample["label"] = [r["label"] for r in results]
sample["score"] = [r["score"] for r in results]
sample[["star_rating", "label", "score"]].head(10)

,star_rating,label,score
40836,1,NEGATIVE,0.999807
85497,1,NEGATIVE,0.990978
124434,1,NEGATIVE,0.998919
118528,1,NEGATIVE,0.999704
13480,1,NEGATIVE,0.999466
47179,1,NEGATIVE,0.999234
130193,1,NEGATIVE,0.999613
126902,1,NEGATIVE,0.999815
58985,1,NEGATIVE,0.999799
22608,1,NEGATIVE,0.999736


**About how long did 500 reviews take? At that rate, how long would all 145,000 take?**

**What does `truncation=True` protect against?**

500 reviews took about **11 seconds** (the cell ran from 14:36:51 to 14:37:02). At that rate (~0.02 s per review), all 145,000 reviews would take about 145,000 / 500 × 11 s ≈ 3,200 s, or **roughly 50 minutes to an hour**. `truncation=True` protects against reviews that are longer than the model's maximum input length (512 tokens for DistilBERT): without it, a review of thousands of words would make the model throw an error. With it, the extra text past the limit is cut off and only the first 512 tokens are classified.

## Part 3: Is the Model Any Good?

### Step 11: Create the answer key and score the model

In [10]:
def stars_to_sentiment(stars):
    """Convert a star rating into the sentiment the customer expressed."""
    if stars <= 2:
        return "NEGATIVE"
    if stars >= 4:
        return "POSITIVE"
    return "NEUTRAL"


sample["truth"] = sample["star_rating"].apply(stars_to_sentiment)
graded = sample[sample["truth"] != "NEUTRAL"]
print("Accuracy:", (graded["label"] == graded["truth"]).mean())

Accuracy: 0.8775


**Why do we leave out the 3-star reviews here?**

A 3-star rating is neutral/mixed, so there is no correct POSITIVE or NEGATIVE answer for it. The model can only say one of those two, so any 3-star review would be unfair to grade either way.

**What's the accuracy? Would you trust this model in a product?**

The accuracy is **0.8775 (about 88%)** on the 400 graded reviews (1, 2, 4 and 5 stars). That's decent but I wouldn't fully trust it in a product: it is wrong on roughly 1 in 8 reviews, and it never says "neutral" or gives a star rating. It would be fine for rough trends but not for anything where each wrong call matters.

### Step 12: Break it down by star rating

In [11]:
pd.crosstab(sample["star_rating"], sample["label"])

label,NEGATIVE,POSITIVE
star_rating,,
1,96,4
2,95,5
3,66,34
4,23,77
5,17,83


**Which star rating does the model get wrong most often?**

Among the graded ratings, **4 stars**: 23 of 100 were labeled NEGATIVE (23% wrong), compared to 17% for 5 stars, 5% for 2 stars and 4% for 1 star. The model is much better at negative reviews than at mildly positive ones. (The 3-star reviews are the most split, but they have no right answer.)

**The model only knows POSITIVE and NEGATIVE. What did it do with the 3-star reviews?**

It forced them into one of the two: 66 of the 100 were labeled NEGATIVE and 34 POSITIVE, so it leans negative on lukewarm reviews.

### Step 13: Read the mistakes

In [12]:
mistakes = graded[graded["label"] != graded["truth"]]
for review in mistakes.head(5).itertuples():
    print(review.star_rating, review.label, round(review.score, 2))
    print(review.review_body[:300], "\n")

1 POSITIVE 0.99
Don't get me wrong, the game concept is awesome. Graphics are great.  BUT  If you don't want to lose your progress when the Simcity Servers stop responding, don't buy this. There is no option to save locally just saved to their server so the smallest error will lose your work.  Overall it's a good g 

1 POSITIVE 1.0
Enjoys game 

1 POSITIVE 0.73
I wasted $7. im sure its a good game but i cant even get to the game part. i tried. a lot. 

1 POSITIVE 1.0
Dose not work right on PSVITA so was not happy but it is a fun game love twisted metal games. 

2 POSITIVE 0.87
I bought this game for my 8 year old daughter who likes the kingdom of keflings game on xbox. The graphics are very simple and it is a basic strategy/building style game. Unfortunately, it looks like it would have been cutting edge 15 to 20 years ago, but it really didn't entertain my kids for more  



**For each of the five, why do you think the model got it wrong? Was it the model, or was the customer's rating misleading?**

1. **SimCity (1 star, POSITIVE 0.99):** the review opens with praise ("the game concept is awesome. Graphics are great") before the real complaint about losing progress. The model was fooled by the positive start, so this is the model's mistake (mixed review).
2. **"Enjoys game" (1 star, POSITIVE 1.0):** the text really is positive, so the customer's rating is misleading (likely a mis-click or a complaint about something else like delivery).
3. **"I wasted $7..." (1 star, POSITIVE 0.73):** the customer is clearly unhappy, but the phrase "im sure its a good game" tricked the model; the low confidence (0.73) shows it was unsure. This is the model's mistake.
4. **PSVITA (1 star, POSITIVE 1.0):** mixed review ("was not happy but it is a fun game love twisted metal games"). The rating is arguably misleading, since the text is partly positive and the model followed the positive part.
5. **Kingdom of Keflings (2 stars, POSITIVE 0.87):** the review starts with neutral/positive description and criticism comes later ("Unfortunately..."); the cut-off text is mostly descriptive, so this is a model mistake on a mildly negative review.

Overall: a mix of both. The model struggles with mixed reviews and sarcasm/"but" structure, and sometimes the star rating doesn't match what the customer wrote.

## Part 4: A Model That Predicts Stars

### Step 14: Predict the stars

In [13]:
star_model = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
)
results = star_model(sample["review_body"].tolist(), truncation=True)
sample["pred_stars"] = [int(r["label"][0]) for r in results]  # "4 stars" -> 4

print("Exact:", (sample["pred_stars"] == sample["star_rating"]).mean())
off_by = (sample["pred_stars"] - sample["star_rating"]).abs()
print("Within one star:", (off_by <= 1).mean())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Exact: 0.574
Within one star: 0.912


**How often does it get the exact star rating? How often within one star?**

The star model gets the **exact** star rating **57.4%** of the time and is **within one star 91.2%** of the time.

**Your team is building a sentiment dashboard for a game publisher. Which of the two models would you use, and why?**

I would use the **star-prediction model (nlptown)** for the dashboard. It gives a 1–5 scale instead of just positive/negative, so the publisher can see how unhappy or happy customers are and track averages over time, and it's within one star on 91% of reviews, so its mistakes are usually small. The tradeoff is that it only gets the exact rating about 57% of the time, while the DistilBERT model is about 88% right on the simpler positive/negative question. If the dashboard only needed a positive vs. negative split, DistilBERT would be the safer pick, but since it can't handle neutral or mixed reviews, I'd still go with the star model.

## Part 5: Zero-Shot – What Are People Complaining About?

### Step 15: Classify 20 one-star reviews

In [14]:
zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

labels = ["price", "bugs or crashes", "account or login", "gameplay", "download or install"]
results = zsc(complaints["review_body"].tolist(), candidate_labels=labels)
complaints["topic"] = [r["labels"][0] for r in results]
complaints["topic"].value_counts()

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

topic
download or install    5
bugs or crashes        5
account or login       4
price                  3
gameplay               3
Name: count, dtype: int64

**What do customers complain about most?**

The top topics are **download or install (5)** and **bugs or crashes (5)**, then account or login (4), price (3) and gameplay (3). So technical problems (not working, installing, crashing) make up about half of the complaints.

In [15]:
for text, topic in zip(complaints["review_body"].head(3), complaints["topic"].head(3)):
    print(topic, "-", text[:300], "\n")

price - Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get VC yea! not). 

download or install - I too installed and tried to play Cobi Treasure Island. My PC does meet the proper requirements so I do not know what the problem is. 

gameplay - If you liked \\"Heavy Rain\\", you'll like this.  However, if you'd like for your games to have a faster pace than 'the English Patient', you'll want to skip this title in favor of something more exciting, like \\"Parking Simulator 4\\". 



**Print three of the reviews with their topics. Does the model's choice make sense?**

Partly. The Cobi Treasure Island review ("download or install") makes sense, since the person couldn't get the game to run. The "Heavy Rain" / "Parking Simulator 4" review ("gameplay") also makes sense, because it complains about the slow pace. The first review was labeled "price", which is only partly right: it mentions paying $30 extra, but it is mostly about the confusing interface and gameplay, so the model picked a keyword-like match and "gameplay" would have been better.

### Step 16: Change the labels

In [16]:
more_labels = labels + ["technical problems", "HR policy"]
new_results = zsc(complaints["review_body"].head(5).tolist(), candidate_labels=more_labels)
for old, new in zip(results[:5], new_results):
    print("Before:", old["labels"][0], round(old["scores"][0], 2))
    print("After: ", new["labels"][0], round(new["scores"][0], 2), "\n")

Before: price 0.66
After:  price 0.59 

Before: download or install 0.41
After:  download or install 0.3 

Before: gameplay 0.36
After:  gameplay 0.32 

Before: bugs or crashes 0.53
After:  technical problems 0.7 

Before: download or install 0.79
After:  download or install 0.45 



**What happened to the top labels and their scores?**

**The scores for each review add up to 1. How does that explain what you saw?**

**What does this tell you about choosing labels for a real product?**

**What happened to the top labels and their scores?** For four of the five reviews the top label stayed the same but its score dropped (price 0.66 → 0.59, download or install 0.41 → 0.30, gameplay 0.36 → 0.32, download or install 0.79 → 0.45). For the fifth review, the top label changed from "bugs or crashes" (0.53) to "technical problems" (0.70), the new overlapping label.

**How does the sum-to-1 explain it?** The scores for a review always add up to 1, so they're shares of a fixed pool. Adding two labels means the probability has to be split among seven labels instead of five, so the existing labels lose some. The overlapping label ("technical problems") took a big share from "bugs or crashes" and "download or install", while the irrelevant one ("HR policy") just took a little away.

**What does this tell you about choosing labels for a real product?** The labels need to be distinct and not overlapping, because similar labels split the score and can flip the top answer; irrelevant labels also dilute the scores. The scores are relative to the label set you choose, not absolute confidence, so I'd choose a clear, mutually exclusive set of labels and test it on real reviews before trusting it.